# WeakTr VOC reproduction on Kaggle

Use one GPU. Prepare packages and copy data with the accelerator disabled. Candidate environment: compatibility has **not** been verified on Kaggle. Training cells default to disabled. Run each stage deliberately; preserve 12 hours for recovery/final evaluation. Colab spending is not authorized.

Clone `https://github.com/ntmphu/WeakTr.git`, branch `reproduction/voc-deit-small`, into `/kaggle/working/WeakTr` before setup. Attach VOC data and official ImageNet DeiT-S weights. Read `docs/KAGGLE.md` for acceptance, storage and transfer steps.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/kaggle/working/WeakTr')
ENV = Path('/kaggle/working/weaktr-env')
PYTHON = ENV / 'bin/python'
DATA = Path('/kaggle/working/data')  # contains voc12/VOCdevkit/VOC2012
OUTPUT = Path('/kaggle/working/weaktr-artifacts')
PRETRAINED = Path('/kaggle/input/weaktr-inputs/deit_small_patch16_224.pth')
RESUME_PHASE1 = ''  # restored local checkpoint.pth
RESUME_PHASE2 = ''  # OUTPUT/phase2/checkpoint.pth after restoring the whole directory
SEED = 504
assert (REPO / 'AGENTS.md').is_file()
def run(argv, cwd=REPO):
    subprocess.run([str(a) for a in argv], cwd=cwd, check=True)
def stage(name, *extra):
    run([PYTHON, '-m', 'reproduction.launch', name, '--data', DATA,
         '--output', OUTPUT, '--pretrained', PRETRAINED, '--seed', SEED, *extra])

## CPU setup

This creates an isolated Python 3.10 environment. Enable Internet for setup; keep the accelerator disabled. Review the installation output, then restart this notebook's parameter cell if needed. `mmcv` is the CPU package: operator compatibility still needs runtime acceptance. If imports fail, resolve them and record the environment before enabling long training.

In [ ]:
INSTALL = False
if INSTALL:
    run([sys.executable, '-m', 'pip', 'install', 'uv==0.5.11'])
    run([sys.executable, '-m', 'uv', 'venv', '--python', '3.10', ENV])
    run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
         'torch==1.13.1+cu117', 'torchvision==0.14.1+cu117',
         '--extra-index-url', 'https://download.pytorch.org/whl/cu117'])
    run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
         '-r', REPO / 'reproduction/requirements-kaggle.txt', 'Cython==0.29.36', 'setuptools<70', 'wheel'])
    run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
         '--no-build-isolation', 'git+https://github.com/lucasb-eyer/pydensecrf.git'])
    # pip freeze records the resolved pydensecrf source commit.
    OUTPUT.mkdir(parents=True, exist_ok=True)
    with (OUTPUT / 'environment-freeze.txt').open('w') as stream:
        subprocess.run([sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', str(PYTHON)], stdout=stream, check=True)

## Data preparation and preflight

Copy attached datasets onto runtime-local `DATA` before proceeding; ensure sufficient disk space. No dataset download or GPU use happens here. See upstream `docs/prepare_dataset.md`. Preflight checks 10,582 training and 1,449 validation IDs, disjointness, image/mask integrity, label IDs and the actual mmseg split files. It also verifies the official ImageNet checkpoint's published hash prefix. Ground-truth masks remain available for evaluation and official CAM checkpoint selection.

In [ ]:
RUN_PREFLIGHT = False
if RUN_PREFLIGHT:
    stage('preflight')
    run([PYTHON, '-m', 'pytest', 'tests/test_reproduction.py', '-q'])
# Inspect commands before launching:
if PYTHON.is_file():
    stage('phase1', '--dry-run')
    stage('phase2', '--dry-run')
else:
    print('Run CPU setup before reviewing commands in the isolated environment.')

## GPU compatibility and reference evaluation

Enable one Kaggle GPU only for this section onward. Smoke checks run both architectures at official input dimensions with synthetic batch size 1, ImageNet initialization, Phase-1 AMP and Phase-2 gradient clipping. They do not establish official batch-size memory acceptance, recovery equivalence, or author-checkpoint metrics. Record those gates separately in `docs/REPRODUCTION_PLAN.md`.

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    run([PYTHON, '-m', 'reproduction.smoke', '--pretrained', PRETRAINED,
         '--record', OUTPUT / 'gpu-smoke.json'])

Author reference checkpoints must come from the upstream README. Keep them in a separate output directory, with source URLs and SHA256 recorded. For Phase 1, evaluate author CAMs on `voc12/train_id.txt` using scale 1.0 and thresholds 40–59. For Phase 2, place its matching `variant.yml` beside the checkpoint and invoke the evaluation command in `docs/KAGGLE.md`. These are reference evaluations; generated reference masks must not become reproduction training targets.

## Pilot and budget forecast

Run one complete official Phase-1 epoch, including scheduled validation and train-set CAM selection. The stop option preserves the 60-epoch scheduler. Reuse its checkpoint for long training. Run a Phase-2 pilot after your pseudo-labels exist, preserving its 100-epoch schedule. Measure CAM/CRF throughput on a representative subset in a separate output directory. Forecast includes selection/evaluation costs and all remaining stages.

In [ ]:
RUN_PHASE1_PILOT = False
if RUN_PHASE1_PILOT:
    stage('phase1', '--stop-after-epoch', '1')
# Record measured seconds below. Phase-2 estimates must be identified until measured.
AVAILABLE_HOURS = None  # check actual Kaggle allocation; reported total ceiling is 60
SPENT_HOURS = 0.0
PHASE1_EPOCH_SECONDS = None  # includes classification and CAM selection
PHASE2_EPOCH_SECONDS = None  # includes validation; initially a documented conservative estimate
CAM_AND_CRF_HOURS = None
FORECAST_VERIFIED = False
if all(v is not None for v in [AVAILABLE_HOURS, PHASE1_EPOCH_SECONDS, PHASE2_EPOCH_SECONDS, CAM_AND_CRF_HOURS]):
    remaining = 59 * PHASE1_EPOCH_SECONDS / 3600 + 100 * PHASE2_EPOCH_SECONDS / 3600 + CAM_AND_CRF_HOURS
    print({'forecast_hours': remaining, 'available_after_12h_reserve': min(60, AVAILABLE_HOURS) - SPENT_HOURS - 12})
    FORECAST_VERIFIED = remaining <= min(60, AVAILABLE_HOURS) - SPENT_HOURS - 12
# Set only after measured acceptance and reference evaluation have been recorded.
GPU_ACCEPTANCE_RECORDED = False

## Phase 1: full training

Execute deliberately after the budget gate. When resuming a pilot or a saved Kaggle session, set `RESUME_PHASE1` to its verified full snapshot. Do not change total epochs, seed, batch size or selection settings. An interruption during an epoch replays that epoch from the latest completed snapshot.

In [ ]:
RUN_PHASE1 = False
if RUN_PHASE1:
    assert FORECAST_VERIFIED and GPU_ACCEPTANCE_RECORDED
    extra = ['--resume', RESUME_PHASE1] if RESUME_PHASE1 else []
    stage('phase1', *extra)

## Generate CAMs and CRF pseudo-labels

Use the reproduction's `checkpoint_best_mIoU.pth`, scales 1.0/1.2, all training IDs, and threshold 43 (0.43). This can take GPU time; time it separately. Mask validation runs after generation.

In [ ]:
RUN_CAMS = False
RUN_PSEUDO = False
if RUN_CAMS:
    stage('cams')
if RUN_PSEUDO:
    stage('pseudo')

## Phase 2: pilot and full training

The launcher explicitly selects `OUTPUT/pseudo` as training annotations and fingerprints their contents for resume compatibility. It checks both data and split files before training. Preserve evaluation every epoch, batch size 4, crop/window 480, resize 520, stride 320, SGD/poly schedule, encoder LR multiplier 0.1 and gradient-clipping settings 1.2/120.

In [ ]:
RUN_PHASE2_PILOT = False
RUN_PHASE2 = False
if RUN_PHASE2_PILOT:
    stage('phase2', '--stop-after-epoch', '1')
if RUN_PHASE2:
    assert FORECAST_VERIFIED and GPU_ACCEPTANCE_RECORDED
    extra = ['--resume', RESUME_PHASE2] if RESUME_PHASE2 else []
    stage('phase2', *extra)

## Final evaluation

Upstream final evaluation uses the latest `phase2/checkpoint.pth`. Save raw multiscale and post-CRF scores separately. `scores_ms.yml` contains raw per-class IoU. CRF stdout contains post-CRF per-class scores; preserve it. Compare against 74.0% only after completing both schedules. Record selected qualitative figures and all deviations.

In [ ]:
RUN_EVALUATION = False
RUN_FINAL_CRF = False
if RUN_EVALUATION:
    stage('evaluate')
if RUN_FINAL_CRF:
    stage('crf')

## Preserve and restore

Stop at epoch boundaries. Preserve the complete `OUTPUT` tree, configs, package freeze and logs in saved Kaggle notebook-version outputs. Active runtime disk alone is not durable. Download a backup to your local machine; optional Drive archiving happens from there. Verify hashes in a fresh runtime before resuming. Never put datasets, weights or generated mask/CAM collections in Git.

In [ ]:
WRITE_TRANSFER_MANIFEST = False
VERIFY_RESTORED_OUTPUTS = False
if WRITE_TRANSFER_MANIFEST:
    run([PYTHON, '-m', 'reproduction.preflight', '--transfer-dir', OUTPUT])
if VERIFY_RESTORED_OUTPUTS:
    run([PYTHON, '-m', 'reproduction.preflight', '--transfer-dir', OUTPUT, '--verify-transfer'])